In [24]:



import sys
from pathlib import Path



# adjust if your path differs
PROJECT_ROOT = Path.home() / "PycharmProjects" / "Magisterka"
PKG_ROOT = PROJECT_ROOT / "nanoGPT-20251228T135841Z-3-001"
sys.path.insert(0, str(PKG_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))
print(sys.path)
sys.path.insert(0, str(PROJECT_ROOT))
from nanoGPT.model import GPTConfig, GPT
print(GPT)
#from nanoGPT.sample import temperature


from praca_magisterska.v1.Scoring import *

'''
@torch.no_grad()
def nanogpt_generate(
    out_dir: str,
    prompt: str,
    stop_char: Optional[str] = None,
    *,
    max_new_tokens: int = 200,
    temperature: float = 0.8,
    top_k: int = 200,
    device: Optional[str] = None,
    dtype: Optional[str] = None,
    compile_model: bool = False,
    seed: int = 1337,
) -> str:
    """
    Generate text from a nanoGPT checkpoint in `out_dir`.

    Args:
        out_dir: Directory containing ckpt.pt (and optionally config info for dataset).
        prompt: Prompt string.
        stop_char: Optional single-character stop condition (e.g. '\\n'). If provided,
                   generation stops at the first occurrence of stop_char in the newly generated text.
        max_new_tokens: Maximum number of new tokens to generate.
        temperature: Sampling temperature.
        top_k: Top-k filtering for sampling.
        device: 'cpu', 'cuda', 'cuda:0', etc. If None, auto-select.
        dtype: 'float32' | 'float16' | 'bfloat16'. If None, auto-select like nanoGPT sample.py.
        compile_model: Whether to torch.compile the model.
        seed: RNG seed.

    Returns:
        Generated continuation (ONLY the new text after the prompt).
    """
    if stop_char is not None and (not isinstance(stop_char, str) or len(stop_char) != 1):
        raise ValueError("stop_char must be a single character (e.g. '\\n') or None.")

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    device_type = "cuda" if "cuda" in device else "cpu"

    if dtype is None:
        if device_type == "cuda" and torch.cuda.is_available() and torch.cuda.is_bf16_supported():
            dtype = "bfloat16"
        elif device_type == "cuda":
            dtype = "float16"
        else:
            dtype = "float32"

    ptdtype = {"float32": torch.float32, "bfloat16": torch.bfloat16, "float16": torch.float16}[dtype]
    ctx = (
        torch.amp.autocast(device_type=device_type, dtype=ptdtype)
        if device_type == "cuda" and dtype != "float32"
        else torch.autocast("cpu")  # no-op-ish fallback; we won't rely on it
    )

    # Reproducibility
    torch.manual_seed(seed)
    if device_type == "cuda":
        torch.cuda.manual_seed(seed)
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

    ckpt_path = os.path.join(out_dir, "ckpt.pt")
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")

    # Load checkpoint (PyTorch 2.6+ may default weights_only=True, so set False)
    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)

    # Build model
    if "model_args" not in checkpoint or "model" not in checkpoint:
        raise ValueError("ckpt.pt doesn't look like a nanoGPT checkpoint (missing 'model_args'/'model').")

    gptconf = GPTConfig(**checkpoint["model_args"])
    model = GPT(gptconf)

    state_dict = checkpoint["model"]
    # Handle torch.compile prefix
    unwanted_prefix = "_orig_mod."
    for k in list(state_dict.keys()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)

    model.load_state_dict(state_dict)
    model.eval()
    model.to(device)

    if compile_model:
        model = torch.compile(model)

    # Choose encoding/decoding:
    # - If this is a resume checkpoint and it has config['dataset'], try meta.pkl for char-level.
    encode = decode = None
    meta = None

    dataset = None
    if isinstance(checkpoint.get("config"), dict):
        dataset = checkpoint["config"].get("dataset")

    if dataset is not None:
        meta_path = os.path.join("data", dataset, "meta.pkl")
        if os.path.exists(meta_path):
            with open(meta_path, "rb") as f:
                meta = pickle.load(f)
            stoi, itos = meta["stoi"], meta["itos"]

            def encode(s: str):
                # For unknown chars, you can decide to raise or map to something.
                # Here we raise to avoid silent corruption.
                try:
                    return [stoi[c] for c in s]
                except KeyError as e:
                    raise ValueError(f"Prompt contains unknown character not in vocab: {e.args[0]!r}")

            def decode(ids):
                return "".join(itos[i] for i in ids)

    if encode is None or decode is None:
        # GPT-2 BPE fallback
        enc = tiktoken.get_encoding("gpt2")
        encode = lambda s: enc.encode(s, allowed_special={"<|endoftext|>"})
        decode = lambda ids: enc.decode(ids)

    # Encode prompt
    prompt_ids = encode(prompt)
    x = torch.tensor(prompt_ids, dtype=torch.long, device=device)[None, ...]

    # Generate
    with torch.no_grad():
        # For CPU, don't rely on autocast; for CUDA with float16/bf16, autocast helps.
        if device_type == "cuda" and dtype != "float32":
            with torch.amp.autocast(device_type=device_type, dtype=ptdtype):
                y = model.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
        else:
            y = model.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)

    full_text = decode(y[0].tolist())
    continuation = full_text[len(prompt):]  # only newly generated text

    if stop_char is not None:
        idx = continuation.find(stop_char)
        if idx != -1:
            continuation = continuation[:idx]

    return continuation
'''

['/home/lukasz/PycharmProjects/Magisterka', '/home/lukasz/PycharmProjects/Magisterka/nanoGPT-20251228T135841Z-3-001', '/home/lukasz/PycharmProjects/Magisterka/nanoGPT-20251228T135841Z-3-001/nanoGPT', '/home/lukasz/PycharmProjects/Magisterka/nanoGPT-20251228T135841Z-3-001/nanoGPT', '/home/lukasz/PycharmProjects/Magisterka/nanoGPT-20251228T135841Z-3-001/nanoGPT', '/home/lukasz/PycharmProjects/Magisterka/nanoGPT-20251228T135841Z-3-001/nanoGPT', '/home/lukasz/PycharmProjects/Magisterka/nanoGPT-20251228T135841Z-3-001/nanoGPT', '/home/lukasz/PycharmProjects/Magisterka/nanoGPT-20251228T135841Z-3-001/nanoGPT', '/home/lukasz/PycharmProjects/Magisterka', '/home/lukasz/PycharmProjects/Magisterka', '/home/lukasz/PycharmProjects/Magisterka/nanoGPT-20251228T135841Z-3-001', '/snap/pycharm/15/plugins/python-ce/helpers/jupyter_debug', '/snap/pycharm/15/plugins/python-ce/helpers/pydev', '/home/lukasz/PycharmProjects/Magisterka', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-d

'\n@torch.no_grad()\ndef nanogpt_generate(\n    out_dir: str,\n    prompt: str,\n    stop_char: Optional[str] = None,\n    *,\n    max_new_tokens: int = 200,\n    temperature: float = 0.8,\n    top_k: int = 200,\n    device: Optional[str] = None,\n    dtype: Optional[str] = None,\n    compile_model: bool = False,\n    seed: int = 1337,\n) -> str:\n    """\n    Generate text from a nanoGPT checkpoint in `out_dir`.\n\n    Args:\n        out_dir: Directory containing ckpt.pt (and optionally config info for dataset).\n        prompt: Prompt string.\n        stop_char: Optional single-character stop condition (e.g. \'\\n\'). If provided,\n                   generation stops at the first occurrence of stop_char in the newly generated text.\n        max_new_tokens: Maximum number of new tokens to generate.\n        temperature: Sampling temperature.\n        top_k: Top-k filtering for sampling.\n        device: \'cpu\', \'cuda\', \'cuda:0\', etc. If None, auto-select.\n        dtype: \'float3

In [25]:
from dataclasses import dataclass
from typing import Callable
import torch
from model import GPTConfig, GPT


@dataclass
class NanoGPTBundle:
    model: GPT
    encode: Callable[[str], list[int]]
    device: str
    device_type: str
    ptdtype: torch.dtype

from dataclasses import dataclass
from typing import Callable, Optional
import os
import pickle
import torch
import tiktoken
from model import GPTConfig, GPT


@dataclass
class NanoGPTBundle:
    model: GPT
    encode: Callable[[str], list[int]]
    decode_token: Callable[[int], str]         # <-- NOWE: id -> tekst tokenu
    decode_ids: Callable[[list[int]], str]     # <-- NOWE: lista id -> tekst
    device: str
    device_type: str
    ptdtype: torch.dtype


def load_nanogpt_bundle(
    out_dir: str,
    *,
    device: Optional[str] = None,
    dtype: Optional[str] = None,
) -> NanoGPTBundle:
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    device_type = "cuda" if "cuda" in device else "cpu"

    if dtype is None:
        if device_type == "cuda" and torch.cuda.is_bf16_supported():
            dtype = "bfloat16"
        elif device_type == "cuda":
            dtype = "float16"
        else:
            dtype = "float32"

    ptdtype = {"float32": torch.float32, "float16": torch.float16, "bfloat16": torch.bfloat16}[dtype]

    ckpt_path = os.path.join(out_dir, "ckpt.pt")
    print(ckpt_path)
    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)

    gptconf = GPTConfig(**checkpoint["model_args"])
    model = GPT(gptconf)

    state_dict = checkpoint["model"]
    unwanted_prefix = "_orig_mod."
    for k in list(state_dict.keys()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)

    model.load_state_dict(state_dict)
    model.eval().to(device)

    # --- encoder/decoder ---
    encode = None
    decode_token = None
    decode_ids = None

    dataset = checkpoint.get("config", {}).get("dataset")
    if dataset:
        meta_path = os.path.join("data", dataset, "meta.pkl")
        if os.path.exists(meta_path):
            with open(meta_path, "rb") as f:
                meta = pickle.load(f)
            stoi, itos = meta["stoi"], meta["itos"]

            def encode(s: str) -> list[int]:
                return [stoi[c] for c in s]

            def decode_token(i: int) -> str:
                return itos[i]

            def decode_ids(ids: list[int]) -> str:
                return "".join(itos[i] for i in ids)

    if encode is None:
        enc = tiktoken.get_encoding("gpt2")

        def encode(s: str) -> list[int]:
            return enc.encode(s, allowed_special={"<|endoftext|>"})

        def decode_token(i: int) -> str:
            return enc.decode([i])

        def decode_ids(ids: list[int]) -> str:
            return enc.decode(ids)

    return NanoGPTBundle(
        model=model,
        encode=encode,
        decode_token=decode_token,
        decode_ids=decode_ids,
        device=device,
        device_type=device_type,
        ptdtype=ptdtype,
    )

bundle = load_nanogpt_bundle("out1")

out1/ckpt.pt
number of parameters: 85.00M


In [26]:
import numpy as np
import torch
from typing import Optional
from tqdm import tqdm


@torch.no_grad()
def next_token_distribution_from_bundle(
    bundle: NanoGPTBundle,
    prompt: str,
    *,
    temperature: float = 1.0,
    top_k: Optional[int] = None,
) -> np.ndarray:
    x = torch.tensor(bundle.encode(prompt), dtype=torch.long, device=bundle.device)[None, :]

    with torch.autocast(
        device_type=bundle.device_type,
        dtype=bundle.ptdtype,
        enabled=(bundle.device_type == "cuda" and bundle.ptdtype != torch.float32),
    ):
        logits, _ = bundle.model(x)

    # logits: [1, T, V]
    next_logits = logits[0, -1, :] / temperature

    if top_k is not None:
        values, indices = torch.topk(next_logits, top_k)
        probs = torch.softmax(values, dim=-1)  # <-- TU powstają prawdopodobieństwa (dla top_k)
        return np.array([(int(i), float(p)) for i, p in zip(indices, probs)], dtype=float)

    probs = torch.softmax(next_logits, dim=-1)  # <-- TU powstają prawdopodobieństwa (dla całego vocab)
    return np.array([(int(i), float(probs[i])) for i in range(probs.shape[0])], dtype=float)



def random_next_token(
        bundle: NanoGPTBundle,
        prompt: str,
        verbose: bool = False,
        temperature: float = 1.0,
        randomize: float = 0.0

):
    source = next_token_distribution_from_bundle(bundle,prompt,temperature=temperature)
    token_ids = source[:,0]
    probs = source[:,1]
    if verbose:
        verbose_table = []
        for i in range(len(token_ids)):
            verbose_table.append([bundle.decode_token(token_ids[i]),probs[i]])
        verbose_table.sort(key=lambda x: x[1],reverse=True)
        for i in verbose_table[:10]:
            print(i)
    return bundle.decode_token(random.choices(token_ids,weights=probs)[0])

prompt = "Prove that: (a ∧ b"
for i in tqdm(range(10)):
    if prompt.endswith(". "):
        prompt+=random_next_token(bundle,prompt,verbose=True,temperature=10)
    prompt+=random_next_token(bundle,prompt,verbose=True,temperature=1)
    print(prompt)
    print("\n")


def last_token_probability_from_bundle(
    bundle: NanoGPTBundle,
    prompt: str,
    *,
    temperature: float = 1.0,
    top_k: Optional[int] = None,
):
    distribution = next_token_distribution_from_bundle(bundle, prompt[:-1], temperature=temperature, top_k=top_k)
    key = bundle.encode(prompt[-1])[0]
    return distribution[key][1]
prompt = "Prove that: (a ∧ c"
last_token_probability_from_bundle(bundle,prompt)

100%|██████████| 10/10 [00:00<00:00, 251.89it/s]

[')', np.float64(1.0)]
[' ', np.float64(6.139278411865234e-06)]
['\n', np.float64(7.497146725654602e-08)]
['-', np.float64(9.837094694375992e-09)]
['h', np.float64(3.346940502524376e-09)]
['i', np.float64(3.14321368932724e-09)]
['1', np.float64(2.750311978161335e-09)]
['e', np.float64(1.3897079043090343e-09)]
['o', np.float64(9.74978320300579e-10)]
['d', np.float64(6.402842700481415e-10)]
Prove that: (a ∧ b)


[' ', np.float64(1.0)]
['s', np.float64(3.702007234096527e-08)]
['(', np.float64(1.30385160446167e-08)]
['d', np.float64(7.683411240577698e-09)]
['¬', np.float64(4.802132025361061e-09)]
['7', np.float64(4.045432433485985e-09)]
['-', np.float64(3.448803909122944e-09)]
['→', np.float64(3.448803909122944e-09)]
['6', np.float64(3.245077095925808e-09)]
[')', np.float64(2.9103830456733704e-09)]
Prove that: (a ∧ b) 


['→', np.float64(0.703125)]
['∨', np.float64(0.29296875)]
['↔', np.float64(0.005035400390625)]
['∧', np.float64(0.00060272216796875)]
[' ', np.float64(0.000125885009765625

np.float64(0.015625)

In [27]:
def extract_variables_from_line(line):
    l = 0
    for i in range(len(line)):
        if line[i] == ".":
            l = i+2
            break
    r = 0
    for j in range(len(line)):
        if line[j] == " " and line[j+1] == " " and line[j+2] == " ":
            r = j
            break
    return variables_order((parse_infix(line[l:r])))

def extract_variables_from_first_line(prompt):
    line = prompt.splitlines()[0]
    variables = variables_order(parse_infix(line[12:]))
    variables = [to_infix(i) for i in variables]
    variables = set(variables)
    return variables

extract_variables_from_first_line("Prove that: (a ∧ b) → (¬a → c)")


{'a', 'b', 'c'}

In [28]:
def next_line(
    bundle: NanoGPTBundle,
    prompt: str,
    *,
    temperature: float = 1.0,
    temperature_begin_line : float = 10,
    multiplier_begin_line : float = 0.66666666,
    top_k: Optional[int] = None
)->str:
    acceptable_variables = extract_variables_from_first_line(prompt)
    ans = prompt
    next_token = None
    while next_token != "\n":
        if ans.endswith(". "):
            next_token = "."
            while not next_token in set(list("(¬⊤⊥")).union(acceptable_variables):
                temp = max(temperature_begin_line * multiplier_begin_line**prompt.count("\n"),temperature)
                next_token = random_next_token(bundle,ans,temperature=temp)
            ans += next_token
        else:
            next_token = random_next_token(bundle,ans,temperature=temperature)
            ans += next_token
    return ans.splitlines()[-1]
next_line(bundle,"Prove that: (a ∧ b) → (¬a → c)\n",temperature=1)


'1. (a ∧ b) → b   assumption'

In [29]:
def is_proof_complete(prompt:str):

    lines = prompt.splitlines()
    proof = ""
    for i in lines[1:]:
        proof += i
        proof += "\n"
    formula = parse_infix(lines[0][12:])
    proof_structure = check_proof(proof)
    if proof_structure == []:
        return False
    return (proof_structure[-1].conclusion == formula) and len(proof_structure[-1].assumptions.formulas) ==0

In [30]:
prompt = """Prove that: (a ∧ b) → (¬¬b ∨ b)
1. a ∧ b   assumption"""
is_proof_complete(prompt)



False

In [31]:
def find_proof_silly(
    bundle: NanoGPTBundle,
    prompt: str,
    *,
    temperature: float = 1.0,
    top_k: Optional[int] = None,
    temperature_begin_line : float = 10,
    multiplier_begin_line : float = 0.66666666
):
    acceptable_variables = variables_order(parse_infix(prompt.splitlines()[0][12:]))
    iteration = 0
    last_valid_iteration = 0
    proof = ""
    old_attempts = []
    while not is_proof_complete(prompt):
        try:
            iteration+=1
            #print(iteration)
            if len(prompt)>900 or (iteration > last_valid_iteration+20):
                #temperature *=1.01
                print("restarting",temperature)
                last_valid_iteration = iteration
                prompt = prompt.splitlines()[0]+"\n"
                old_attempts.append(prompt)
                proof = ""

            line = next_line(bundle,prompt,temperature=temperature,top_k=top_k,temperature_begin_line=temperature_begin_line,multiplier_begin_line=multiplier_begin_line)
            number_of_repeats = 0
            for i in old_attempts:
                if i.startswith(prompt+ line+ "\n"):
                    number_of_repeats+=1
            new_proof = proof+line+"\n"
            #continues = random.choices([True,False],weights=[1/(number_of_repeats+1),number_of_repeats/(number_of_repeats+1)])[0]
            if True:#continues:
                try:
                    line_variables = extract_variables_from_line(line)
                    for i in line_variables:
                        if i not in acceptable_variables:
                            raise Exception("bad variable")
                    check_proof(new_proof)

                    print(new_proof,"\n")
                    proof = new_proof
                    prompt = prompt+ line+ "\n"
                    last_valid_iteration = iteration
                except:
                    continue
        except:
            #temperature *=1.01
            print("restarting",temperature)
            last_valid_iteration = iteration
            prompt = prompt.splitlines()[0]+"\n"
            old_attempts.append(prompt)
            proof = ""

#find_proof_silly(bundle, "Prove that: ¬(p ∧ (¬p ∧ q))\n",temperature_begin_line=1,multiplier_begin_line=1)

In [32]:
#find_proof_little_smarter(bundle, "Prove that: (a ∧ b) → (¬a → c)\n",temperature_begin_line=1,multiplier_begin_line=1)


In [33]:
'''
def find_proof_silly(
        bundle: NanoGPTBundle,
        prompt: str,
        *,
        temperature: float = 1.0,
        top_k: Optional[int] = None,
):
    acceptable_variables = variables_order(parse_infix(prompt.splitlines()[0][12:]))

    iteration = 0
    last_valid_iteration = 0

    proof = ""
    base_prompt = prompt.splitlines()[0] + "\n"

    # "Doświadczenie":
    # 1) najlepszy dotychczasowy stan (żeby restart nie cofał do zera)
    best_prompt = base_prompt
    best_proof = ""
    best_len = 0

    # 2) pamięć złych prób: dla danego prefiksu (prompt) pamiętamy linie, które kończyły się porażką
    bad_next_line_counts: dict[str, dict[str, int]] = {}
    max_resample_tries = 64

    def remember_bad(prefix: str, line: str):
        d = bad_next_line_counts.setdefault(prefix, {})
        d[line] = d.get(line, 0) + 1

    def should_avoid(prefix: str, line: str) -> bool:
        c = bad_next_line_counts.get(prefix, {}).get(line, 0)
        return c >= 1  # im więcej razy porażka, tym bardziej unikamy; tu twarde unikanie po 1 razie

    while not is_proof_complete(prompt):
        iteration += 1

        # restart: wracamy do najlepszego znanego stanu, a nie do zera
        if len(prompt) > 900 or (iteration > last_valid_iteration + 20):
            print("restarting", temperature)
            last_valid_iteration = iteration
            prompt = best_prompt
            proof = best_proof

        # próbkuj linię, ale unikaj tych, które już znamy jako złe dla tego prefiksu
        line = None
        for _ in range(max_resample_tries):
            candidate = next_line(bundle, prompt, temperature=temperature, top_k=top_k)

            try:
                line_variables = extract_variables_from_line(candidate)
                good = True
                for v in line_variables:
                    if v not in acceptable_variables:
                        pass
                if not good:
                    continue
            except Exception:
                continue

            if should_avoid(prompt, candidate):
                continue

            line = candidate
            break

        if line is None:
            # jeśli nie udało się znaleźć sensownej nowej linii, restart do najlepszego stanu
            print("restarting", temperature)
            last_valid_iteration = iteration
            prompt = best_prompt
            proof = best_proof
            continue

        new_proof = proof + line + "\n"

        try:
            check_proof(new_proof)
            print(new_proof, "\n")
            proof = new_proof
            prompt = prompt + line + "\n"
            last_valid_iteration = iteration

            if len(proof) > best_len:
                best_len = len(proof)
                best_prompt = prompt
                best_proof = proof

        except Exception:
            # zapamiętaj, że ta linia dla tego prefiksu prowadzi do porażki
            remember_bad(prompt, line)
            continue

'''
#find_proof_silly(bundle, "Prove that: ¬(p ∧ (¬p ∧ q))\n",temperature_begin_line=2)


'\ndef find_proof_silly(\n        bundle: NanoGPTBundle,\n        prompt: str,\n        *,\n        temperature: float = 1.0,\n        top_k: Optional[int] = None,\n):\n    acceptable_variables = variables_order(parse_infix(prompt.splitlines()[0][12:]))\n\n    iteration = 0\n    last_valid_iteration = 0\n\n    proof = ""\n    base_prompt = prompt.splitlines()[0] + "\n"\n\n    # "Doświadczenie":\n    # 1) najlepszy dotychczasowy stan (żeby restart nie cofał do zera)\n    best_prompt = base_prompt\n    best_proof = ""\n    best_len = 0\n\n    # 2) pamięć złych prób: dla danego prefiksu (prompt) pamiętamy linie, które kończyły się porażką\n    bad_next_line_counts: dict[str, dict[str, int]] = {}\n    max_resample_tries = 64\n\n    def remember_bad(prefix: str, line: str):\n        d = bad_next_line_counts.setdefault(prefix, {})\n        d[line] = d.get(line, 0) + 1\n\n    def should_avoid(prefix: str, line: str) -> bool:\n        c = bad_next_line_counts.get(prefix, {}).get(line, 0)\n    

In [34]:
print(LittleProblem(Context(parse_infix("a"),parse_infix("c"),parse_infix("b")),parse_infix("c")))==print(LittleProblem(Context(parse_infix("c"),parse_infix("b"),parse_infix("a")),parse_infix("c")))

'(b), '(c), '(a) ⊢ '(c)
'(b), '(c), '(a) ⊢ '(c)


True

In [35]:
next_line(bundle,"Prove that: (a ∧ b) → (¬a → c)\n",temperature=1)

'1. ¬¬(a ∧ b)   assumption'

In [36]:
prompt = '''Prove that: (a ∧ b) → (¬a → c)
1. a ∧ b   assumption
2. ¬a   assumption
3. a    ∧-elimination, 1
4. ⊥    ¬-elimination, 2, 3'''
random_next_token(bundle,prompt)
#find_proof_silly(bundle, "Prove that: (a ∧ b) → (¬a → c)\n",multiplier_begin_line=0.666666,temperature_begin_line=10)
last_token_probability_from_bundle(bundle,"ab")

np.float64(2.3692846298217773e-06)

In [37]:
def prompt_probability_normalised(bundle, prompt,temperature=1,top_k=None):
    temporal_prompt = prompt.splitlines()[0] + "\n"
    ans_unnormalised = 0
    proof = ""
    for i in prompt.splitlines()[1:]:
        proof += i+"\n"
    for i in proof:
        temporal_prompt += i
        #print(temporal_prompt)
        ans_unnormalised  += np.log(last_token_probability_from_bundle(bundle,temporal_prompt,temperature=temperature,top_k=top_k))
        #print(ans_unnormalised)
    return ans_unnormalised/len(proof)

print(prompt)
prompt_probability_normalised(bundle,prompt)


Prove that: (a ∧ b) → (¬a → c)
1. a ∧ b   assumption
2. ¬a   assumption
3. a    ∧-elimination, 1
4. ⊥    ¬-elimination, 2, 3


np.float64(-0.08406220861980321)

In [45]:
def choose_prompt(prompts,probabilities):
    first_probability = np.sum(probabilities[1:])/len(probabilities[1:])
    probabilities -= np.min(probabilities)
    probabilities += 1/len(probabilities)
    probabilities /= np.sum(probabilities)
    #print(probabilities)
    return random.choices(prompts,weights=probabilities)[0]
def soft_max(x):
    return np.exp(x)/np.sum(np.exp(x))
def find_proof_little_smarter(
    bundle: NanoGPTBundle,
    prompt: str,
    *,
    temperature: float = 1.0,
    top_k: Optional[int] = None,
    max_iter: int = 10000
):
    acceptable_variables = variables_order(parse_infix(prompt.splitlines()[0][12:]))
    prompts = [prompt]
    probabilities_wild = [1.0]
    exponent = [0]
    probabilities = np.array(probabilities_wild)
    I = 0
    while (not is_proof_complete(prompt)) and I < max_iter:
        I += 1
        print(I)
        try:
            prompt_to_expand = np.random.choice(prompts,p=probabilities)
            line = next_line(bundle,prompt_to_expand,temperature=temperature,top_k=top_k,temperature_begin_line=10,multiplier_begin_line=0.66)
            new_prompt = prompt_to_expand+line+"\n"
            if (new_prompt not in prompts) and new_prompt.__len__() < 950:
                try:
                    line_variables = extract_variables_from_line(line)
                    for i in line_variables:
                        if i not in acceptable_variables:
                            raise Exception("bad variable")
                    lines = new_prompt.splitlines()
                    proof = ""
                    for i in lines[1:]:
                        proof += i+"\n"
                    proof = proof[:-1]
                    proof_structure = check_proof(proof)
                    for i in proof_structure[:-1]:
                        if i == proof_structure[-1]:
                            raise Exception("redundant")
                    prompts.append(new_prompt)
                    exponent.append(0)
                    probabilities_wild.append(prompt_probability_normalised(bundle,new_prompt))
                    probabilities = np.array(probabilities_wild)
                    probabilities[0] = probabilities.mean()
                    probabilities = soft_max(probabilities)
                    print(new_prompt)
                    print("\n")
                    prompt = new_prompt
                    print("ok")
                except Exception:
                    prompt = prompts[0]
            else:
                if len(new_prompt.splitlines())>1:
                    idx = 0
                    for i in range(len(prompts)):
                        if prompts[i]==new_prompt:
                            idx = i
                            break
                    exponent[idx] += 1
                    probabilities_wild[idx] *= (0.9 ** exponent[idx])
                    probabilities = soft_max(probabilities)
        except Exception:
            pass
    return prompts
#find_proof_silly(bundle,"Prove that: (a ∧ b) → (¬a → b)\n")
prompts = find_proof_little_smarter(bundle, "Prove that: (a ∧ b) → (¬a → b)\n",max_iter=1000)
#find_proof_little_smarter(bundle, "Prove that: ¬(p ∧ (¬p ∧ q))\n",temperature_begin_line=1,multiplier_begin_line=1)


1
Prove that: (a ∧ b) → (¬a → b)
1. b   assumption



ok
2
Prove that: (a ∧ b) → (¬a → b)
1. a ∧ b   assumption



ok
3
Prove that: (a ∧ b) → (¬a → b)
1. a ∧ b   assumption
2. ¬a   assumption



ok
4
Prove that: (a ∧ b) → (¬a → b)
1. a ∧ b   assumption
2. ¬a   assumption
3. a    ∧-elimination, 1



ok
5
Prove that: (a ∧ b) → (¬a → b)
1. a ∧ b   assumption
2. ¬a   assumption
3. a    ∧-elimination, 1
4. ⊥    ¬-elimination, 2, 3



ok
6
7
Prove that: (a ∧ b) → (¬a → b)
1. a ∧ b   assumption
2. ¬a   assumption
3. a    ∧-elimination, 1
4. ⊥    ¬-elimination, 2, 3
5. a    RAA, 2–4



ok
8
9
Prove that: (a ∧ b) → (¬a → b)
1. a ∧ b   assumption
2. ¬a   assumption
3. a    ∧-elimination, 1
4. ⊥    ¬-elimination, 2, 3
5. a    RAA, 2–4
6. ⊥    ¬-elimination, 2, 5



ok
10
11
Prove that: (a ∧ b) → (¬a → b)
1. a ∧ b   assumption
2. ¬a   assumption
3. a    ∧-elimination, 1
4. ⊥    ¬-elimination, 2, 3
5. a    RAA, 2–4
6. ⊥    ¬-elimination, 2, 5
7. ⊤    ⊥-elimination, 6



ok
12
13
Prove that: (a ∧ b)

In [43]:
print(prompts)

NameError: name 'prompts' is not defined

In [ ]:
for i in range(100):
    print(next_line(bundle,"Prove that: (a ∧ b) → (¬a → b)\n",temperature_begin_line=1,multiplier_begin_line=5))

In [ ]:
def softmax(x):
    return np.exp(x)/sum(np.exp(x))





In [ ]:
choose_prompt([1,2,3],np.array([-1,-1.4,-1.4]))

In [ ]:
[1,2,3,4][:-1]